# Chemical Space Analysis: UMAP with Distance Analysis and QED

This notebook creates a clean UMAP visualization and analyzes distance patterns to R-BIND2 compounds, plus QED drug-likeness analysis.

In [ ]:
TITLE_SIZE = 16
LABEL_SIZE = 14
LEGEND_SIZE = 12
TICK_SIZE = 12

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED
from sklearn.preprocessing import StandardScaler
import umap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('./data/0_input/chem-lib_FDAandInternal.csv')
print(f"Loaded {len(df)} compounds")
print(f"Internal: {len(df[df['type']=='internal'])}, R-BIND2: {len(df[df['type']=='R-BIND2'])}")


In [ ]:
def calculate_optimized_descriptors(smiles_list):
    descriptors = []
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
            
        desc = {
            'MW': Descriptors.MolWt(mol),
            'LogP': Crippen.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'BertzCT': Descriptors.BertzCT(mol),
            'FracCsp3': Descriptors.FractionCSP3(mol),
            'NumAromaticRings': Descriptors.NumAromaticRings(mol),
            'HBD': Descriptors.NumHDonors(mol), 
            'HBA': Descriptors.NumHAcceptors(mol),
            'RotBonds': Descriptors.NumRotatableBonds(mol),
            'NumHeteroatoms': Descriptors.NumHeteroatoms(mol),
            'RingCount': Descriptors.RingCount(mol),
            'MolMR': Crippen.MolMR(mol),
        }
        descriptors.append(desc)
    
    return pd.DataFrame(descriptors)

def calculate_qed_values(smiles_list):
    """Calculate QED (Quantitative Estimate of Drug-likeness) values"""
    qed_values = []
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            qed_values.append(np.nan)
        else:
            qed_values.append(QED.qed(mol))
    
    return qed_values


descriptors_df = calculate_optimized_descriptors(df['SMILES'].tolist())

qed_values = calculate_qed_values(df['SMILES'].tolist())

valid_indices = ~descriptors_df.isnull().any(axis=1)
descriptors_clean = descriptors_df[valid_indices].reset_index(drop=True)
df_clean = df[valid_indices].reset_index(drop=True)
types_clean = df_clean['type']
qed_clean = [qed_values[i] for i in range(len(qed_values)) if valid_indices[i]]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(descriptors_clean)

reducer = umap.UMAP(
    n_neighbors=5,
    min_dist=0.8,
    spread=3.0,
    random_state=123,
    n_components=2,
    metric='euclidean'
)

embedding = reducer.fit_transform(X_scaled)

df_clean['UMAP1-opt'] = embedding[:, 0]
df_clean['UMAP2-opt'] = embedding[:, 1]
df_clean['QED'] = qed_clean

def calculate_distances_to_rbind2(embedding, types):
    """Calculate distance from each internal compound to nearest R-BIND2 compound"""
    internal_mask = types == 'internal'
    rbind2_mask = types == 'R-BIND2'
    
    internal_points = embedding[internal_mask]
    rbind2_points = embedding[rbind2_mask]
    
    min_distances = []
    for i_point in internal_points:
        distances = np.linalg.norm(rbind2_points - i_point, axis=1)
        min_distances.append(np.min(distances))
    
    return np.array(min_distances)

distances_to_rbind2 = calculate_distances_to_rbind2(embedding, types_clean)

print("UMAP coordinates and QED values added to dataframe")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = {
    'internal': '#40E0D0',    # Turquoise
    'R-BIND2': '#FFA500'     # Orange
}

for ctype in ['R-BIND2', 'internal']:  # R-BIND2 first (background)
    mask = types_clean == ctype
    alpha = 0.7 if ctype == 'R-BIND2' else 0.9
    size = 60 if ctype == 'internal' else 45
    zorder = 2 if ctype == 'internal' else 1
    
    label = 'Hit' if ctype == 'internal' else ctype
    
    ax1.scatter(embedding[mask, 0], embedding[mask, 1],
               c=colors[ctype], alpha=alpha, s=size, 
               edgecolors='white', linewidth=0.8, zorder=zorder,
               label=label)

ax1.legend(loc='lower right', fontsize=LEGEND_SIZE)
ax1.set_xlabel('UMAP Dimension 1', fontsize=LABEL_SIZE, fontweight='bold')
ax1.set_ylabel('UMAP Dimension 2', fontsize=LABEL_SIZE, fontweight='bold')
ax1.set_title('Chemical Space Analysis', fontsize=TITLE_SIZE, fontweight='bold')
ax1.grid(False)

for spine in ax1.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)
ax1.set_facecolor('white')

# Distance distribution histogram
ax2.hist(distances_to_rbind2, bins=25, alpha=0.7, color='#2E86AB', edgecolor='black', linewidth=0.8)
ax2.set_xlabel('Distance to Nearest R-BIND2 Compound', fontsize=LABEL_SIZE, fontweight='bold')
ax2.set_ylabel('Number of Internal Compounds', fontsize=LABEL_SIZE, fontweight='bold')
ax2.set_title('Separation from Known RNA Binders', fontsize=TITLE_SIZE, fontweight='bold')
ax2.grid(False)

for spine in ax2.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)
ax2.set_facecolor('white')

'''far_compounds = np.sum(distances_to_rbind2 > np.percentile(distances_to_rbind2, 75))
ax2.text(0.65, 0.85, f'Far compounds: {far_compounds}\n(>75th percentile)', 
         transform=ax2.transAxes, fontsize=11, fontweight='bold',
         bbox=dict(boxstyle="round,pad=0.4", facecolor="lightblue", alpha=0.8))'''

plt.tight_layout()
plt.show()

print("Clean UMAP and distance analysis completed")

In [ ]:
internal_qed = [qed_clean[i] for i in range(len(qed_clean)) if types_clean.iloc[i] == 'internal' and not np.isnan(qed_clean[i])]
rbind2_qed = [qed_clean[i] for i in range(len(qed_clean)) if types_clean.iloc[i] == 'R-BIND2' and not np.isnan(qed_clean[i])]

print(f"Internal compounds QED: {np.mean(internal_qed):.3f} ± {np.std(internal_qed):.3f} (n={len(internal_qed)})")
print(f"R-BIND2 compounds QED: {np.mean(rbind2_qed):.3f} ± {np.std(rbind2_qed):.3f} (n={len(rbind2_qed)})")

# Analyze QED by UMAP position (focus on left vs right)
x_median = np.median(embedding[:, 0])
left_mask = embedding[:, 0] < x_median
right_mask = embedding[:, 0] >= x_median

# Get QED for compounds in different regions
left_rbind2_indices = np.where(left_mask & (types_clean == 'R-BIND2'))[0]
right_rbind2_indices = np.where(right_mask & (types_clean == 'R-BIND2'))[0]
left_internal_indices = np.where(left_mask & (types_clean == 'internal'))[0]
right_internal_indices = np.where(right_mask & (types_clean == 'internal'))[0]

left_rbind2_qed = [qed_clean[i] for i in left_rbind2_indices if not np.isnan(qed_clean[i])]
right_rbind2_qed = [qed_clean[i] for i in right_rbind2_indices if not np.isnan(qed_clean[i])]
left_internal_qed = [qed_clean[i] for i in left_internal_indices if not np.isnan(qed_clean[i])]
right_internal_qed = [qed_clean[i] for i in right_internal_indices if not np.isnan(qed_clean[i])]

print("\nQED by UMAP Region:")
print(f"Left R-BIND2 QED: {np.mean(left_rbind2_qed):.3f} ± {np.std(left_rbind2_qed):.3f} (n={len(left_rbind2_qed)})")
print(f"Right R-BIND2 QED: {np.mean(right_rbind2_qed):.3f} ± {np.std(right_rbind2_qed):.3f} (n={len(right_rbind2_qed)})")
print(f"Left Internal QED: {np.mean(left_internal_qed):.3f} ± {np.std(left_internal_qed):.3f} (n={len(left_internal_qed)})")
print(f"Right Internal QED: {np.mean(right_internal_qed):.3f} ± {np.std(right_internal_qed):.3f} (n={len(right_internal_qed)})")


In [ ]:
# Create QED visualization with UMAP positions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# UMAP colored by QED
scatter = ax1.scatter(embedding[:, 0], embedding[:, 1], c=qed_clean, 
                     cmap='viridis', s=50, alpha=0.8, edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, ax=ax1, label='QED (Drug-likeness)')

ax1.set_xlabel('UMAP Dimension 1', fontsize=LABEL_SIZE, fontweight='bold')
ax1.set_ylabel('UMAP Dimension 2', fontsize=LABEL_SIZE, fontweight='bold')
ax1.set_title('Chemical Space Colored by QED', fontsize=TITLE_SIZE, fontweight='bold')
ax1.grid(False)

for spine in ax1.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)
ax1.set_facecolor('white')

ax2.hist(internal_qed, bins=20, alpha=0.7, label='Internal', color=colors['internal'], edgecolor='black')
ax2.hist(rbind2_qed, bins=20, alpha=0.7, label='R-BIND2', color=colors['R-BIND2'], edgecolor='black')

ax2.set_xlabel('QED (Drug-likeness)', fontsize=LABEL_SIZE, fontweight='bold')
ax2.set_ylabel('Number of Compounds', fontsize=LABEL_SIZE, fontweight='bold')
ax2.set_title('QED Distribution by Compound Type', fontsize=TITLE_SIZE, fontweight='bold')
ax2.legend(fontsize=LEGEND_SIZE)
ax2.grid(False)

for spine in ax2.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)
ax2.set_facecolor('white')

plt.tight_layout()
plt.show()

print("QED analysis visualization completed")

In [ ]:
# Save UMAP results to CSV
output_path = './data/output_from_code/umap_results.csv'
df_clean.to_csv(output_path, index=False)
print(f"Saved UMAP results to {output_path}")